# INCLUDE50 CNN+LSTM Training Pipeline
Fully resumable — safe to re-run any cell after disconnect.

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────
!pip install mediapipe==0.10.31 transformers==4.44.0 timm joblib tqdm scikit-learn -q
print('Done')

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 3: Clone / update repo ───────────────────────────────────────
import os, subprocess

REPO = '/content/Major_Project'
if os.path.exists(REPO):
    os.chdir(REPO)
    !git pull origin branch_03_cnn-and-lstm
else:
    os.chdir('/content')
    !git clone -b branch_03_cnn-and-lstm https://github.com/BishalDubey27/Major_Project.git {REPO}

os.chdir(f'{REPO}/INCLUDE')
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── CELL 4: Configure paths ───────────────────────────────────────────
import os

BASE          = '/content/drive/MyDrive/ISL_Training'
ZIP_DIR       = f'{BASE}/include50_raw_zips'       # folder with all 44 zip files
VIDEOS_DIR    = f'{BASE}/new_include50_videos'     # extracted videos
KEYPOINTS_DIR = f'{BASE}/keypoints'                # mediapipe keypoints
CNN_DIR       = f'{BASE}/cnn_features'             # MobileNetV2 features
SAVE_PATH     = f'{BASE}/trained_model'            # saved .pth files

for d in [VIDEOS_DIR, KEYPOINTS_DIR, CNN_DIR, SAVE_PATH]:
    os.makedirs(d, exist_ok=True)

print('All paths ready')

In [ ]:
# ── CELL 5: Extract zip files (resumable) ────────────────────────────
# Skips zips whose contents are already extracted
import zipfile, os, glob

zips = sorted(glob.glob(f'{ZIP_DIR}/*.zip'))
print(f'Found {len(zips)} zip files')

for i, zp in enumerate(zips):
    name = os.path.splitext(os.path.basename(zp))[0]  # e.g. Adjectives_1of8
    category = name.split('_')[0]                       # e.g. Adjectives
    dest = os.path.join(VIDEOS_DIR, category)

    # Check if already extracted by counting files
    existing = sum(1 for r,d,f in os.walk(dest) for fn in f if fn.lower().endswith('.mov')) if os.path.exists(dest) else 0

    with zipfile.ZipFile(zp, 'r') as z:
        total_in_zip = sum(1 for n in z.namelist() if n.lower().endswith('.mov'))

    if existing >= total_in_zip:
        print(f'[{i+1}/{len(zips)}] {name}: already extracted ({existing} files) — skipping')
        continue

    print(f'[{i+1}/{len(zips)}] {name}: extracting {total_in_zip} files...')
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(VIDEOS_DIR)

total = sum(1 for r,d,f in os.walk(VIDEOS_DIR) for fn in f if fn.lower().endswith('.mov'))
print(f'\nTotal videos: {total}')

In [ ]:
# ── CELL 6: Generate keypoints (resumable) ────────────────────────────
# Skips videos that already have a keypoint JSON file
import os, sys
sys.path.insert(0, '/content/Major_Project/INCLUDE')
os.chdir('/content/Major_Project/INCLUDE')

from generate_keypoints import process_video, load_file

for split in ['train', 'val', 'test']:
    save_dir = f'{KEYPOINTS_DIR}/include50_{split}_keypoints'
    os.makedirs(save_dir, exist_ok=True)

    paths = load_file(f'train_test_paths/include50_{split}.txt', VIDEOS_DIR)
    skipped = processed = errors = 0

    for path in paths:
        label = path.replace('\\', '/').split('/')[-2]
        label = ''.join([c for c in label if c.isalpha()]).lower()
        uid   = '_'.join([label, os.path.splitext(os.path.basename(path))[0]])
        out   = os.path.join(save_dir, f'{uid}.json')

        if os.path.exists(out):
            skipped += 1
            continue
        try:
            process_video(path, save_dir)
            processed += 1
        except Exception as e:
            errors += 1

    total = skipped + processed
    print(f'{split}: {total} total | {processed} new | {skipped} skipped | {errors} errors')

In [ ]:
# ── CELL 7: Extract CNN features (resumable) ──────────────────────────
# MobileNetV2 renders each skeleton frame → 1280-dim feature vector
# Skips files already processed
import sys, os
sys.path.insert(0, '/content/Major_Project/INCLUDE')
os.chdir('/content/Major_Project/INCLUDE')

import argparse
args = argparse.Namespace(
    dataset='include50',
    data_dir=KEYPOINTS_DIR,
    save_dir=CNN_DIR,
    use_cnn=True, use_augs=False,
    model='lstm', transformer_size='small',
    seed=0, batch_size=32, epochs=100,
    learning_rate=1e-4, save_path=SAVE_PATH
)

from cnn_runner import save_cnn_features
save_cnn_features(args)

import glob
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f'{CNN_DIR}/include50_{split}_cnn_features/*.npy'))
    print(f'{split}: {n} CNN feature files')

In [ ]:
# ── CELL 8: Train CNN+LSTM (resumable via best checkpoint) ────────────
import os
os.chdir('/content/Major_Project/INCLUDE')

# Set patience to 20 so training doesn't stop too early
with open('train_nn.py', 'r') as f:
    content = f.read()
content = content.replace('EarlyStopping(patience=10', 'EarlyStopping(patience=20')
with open('train_nn.py', 'w') as f:
    f.write(content)
print('Patience set to 20')

!python runner.py \
    --dataset include50 \
    --model lstm \
    --use_cnn \
    --data_dir {CNN_DIR} \
    --save_path {SAVE_PATH} \
    --epochs 100 \
    --batch_size 32 \
    --learning_rate 1e-4 \
    --use_augs

In [ ]:
# ── CELL 9: Check results ─────────────────────────────────────────────
import torch, glob

pth_files = glob.glob(f'{SAVE_PATH}/*.pth')
print('Saved models:')
for pf in pth_files:
    cp = torch.load(pf, map_location='cpu', weights_only=False)
    print(f'  {os.path.basename(pf)}: val_score={cp.get("score", "N/A")}')

In [ ]:
# ── CELL 10: Download model ───────────────────────────────────────────
from google.colab import files
import glob, os

pth_files = glob.glob(f'{SAVE_PATH}/*.pth')
lstm_models = [f for f in pth_files if 'lstm' in os.path.basename(f).lower()]
target = lstm_models[0] if lstm_models else pth_files[0]

files.download(target)
print(f'Downloaded: {target}')
print('Place this .pth in your INCLUDE/ folder and tell Kiro the filename')